# Modelo de Dicke con corte automático de fotones
Resuelve la ecuación maestra en la base $|n,jm\rangle$, con $\hbar=1$ y $j=N/2$, en el sector colectivo simétrico. Conserva los términos contrarrotantes y la pérdida de fotones; no incluye emisión atómica ni bombeo adicional.

**Ejecuta las celdas en orden.** La primera pide, en este orden: **N, lambda, kappa, m inicial y n inicial**. Enter acepta el ejemplo sugerido. El estado inicial es $|n_0,j,m_0\rangle\langle n_0,j,m_0|$.

El corte `n_max` se calcula automáticamente: una estimación inicial usa los parámetros físicos y luego se aumenta hasta verificar convergencia. El criterio se aplica al intervalo temporal especificado y a los tiempos muestreados; no garantiza precisión para tiempos posteriores. Si no converge dentro del presupuesto de memoria o de iteraciones, el cuaderno se detiene y lo explica.

Requiere `numpy`, `scipy` y `matplotlib`. Si faltan, ejecuta `%pip install numpy scipy matplotlib` en una celda y reinicia el kernel.


## Versión corregida

Ejecuta las celdas en orden en un kernel limpio. Las salidas antiguas se han
retirado para evitar confundir resultados de distintas configuraciones.

La parte cuántica utiliza `resultado_cuantico`. La parte de campo medio
conserva sus propios parámetros, que pueden ser distintos: compara ambos
cálculos solo después de igualar los parámetros y entender sus condiciones
iniciales. Las frecuencias de campo medio no son niveles exactos del
Hamiltoniano para N finito.

La búsqueda de rho estacionaria muestra su tasa de cambio y solo devuelve
una matriz estacionaria cuando cumple el criterio. Si no lo cumple en el
intervalo elegido, informa de ello y permite continuar las secciones
independientes de campo medio.


In [ ]:
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import expm_multiply
import matplotlib.pyplot as plt
import gc

def pedir_entero(texto, defecto, minimo):
    while True:
        try:
            entrada = input(f"{texto} [{defecto}]: ").strip()
            valor = int(entrada) if entrada else defecto
            if valor < minimo:
                raise ValueError
            return valor
        except ValueError:
            print(f"Introduce un entero mayor o igual a {minimo}.")

def pedir_real(texto, defecto, minimo=None):
    while True:
        try:
            entrada = input(f"{texto} [{defecto:g}]: ").strip()
            valor = float(entrada.replace(",", ".")) if entrada else defecto
            if not np.isfinite(valor) or (minimo is not None and valor < minimo):
                raise ValueError
            return valor
        except ValueError:
            print("Introduce un número real finito" +
                  (f" mayor o igual a {minimo}." if minimo is not None else "."))

N = pedir_entero("Número de átomos N", 2, 1)
j = N/2
lam = pedir_real("Acoplamiento lambda", 0.35)
kappa = pedir_real("Pérdida de fotones kappa", 0.10, minimo=0)
while True:
    m_inicial = pedir_real(f"m inicial, entre {-j:g} y {j:g}", -j)
    if (abs(m_inicial) <= j and
            np.isclose(m_inicial+j, round(m_inicial+j), atol=1e-10, rtol=0)):
        m_inicial = -j + round(m_inicial+j)
        break
    print("m debe pertenecer a -j, -j+1, ..., j (semientero si N es impar).")
n_inicial = pedir_entero("Número inicial de fotones", 0, 0)
print(f"N={N}, lambda={lam:g}, kappa={kappa:g}")
print(f"Estado inicial: |n={n_inicial}, j={j:g}, m={m_inicial:g}⟩")
print("n_max se determinará automáticamente en la celda de resolución.")


## Frecuencias, tiempo y precisión
$$H=\omega_0J_z+\omega a^\dagger a+\frac{\lambda}{\sqrt{N}}(a^\dagger+a)(J_++J_-),$$
$$\dot\rho=-i[H,\rho]+\kappa a\rho a^\dagger-\frac{\kappa}{2}\{a^\dagger a,\rho\}.$$

Los cinco valores pedidos no fijan las frecuencias ni el tiempo. Por defecto se usa $\omega=\omega_0=1$, $t\in[0,25]$. Modifícalos aquí antes de resolver. Los parámetros son frecuencias; el tiempo tiene las unidades inversas correspondientes.

Las tolerancias controlan la **convergencia del corte**, no son tolerancias internas de `expm_multiply`. El presupuesto de memoria es una estimación conservadora de almacenamiento y temporales, no una medición de la RAM libre de tu equipo.


In [ ]:
omega = 1.0
omega_0 = 1.0
t_final = 100.0
num_tiempos = 301

# Configuración del ajuste automático; normalmente no hace falta cambiarla.
tolerancia_borde = 1e-7
atol_observables = 1e-6
rtol_observables = 1e-5
tolerancia_distribuciones = 1e-5
tolerancia_rho = 1e-5
niveles_borde = 3
max_iteraciones = 8
presupuesto_memoria_gib = 1.0


# Captura los parámetros cuánticos para evitar depender de otras secciones.
parametros_cuanticos = {
    "N": N, "n_inicial": n_inicial, "m_inicial": m_inicial,
    "omega": omega, "omega_0": omega_0, "lam": lam, "kappa": kappa,
}


## De los elementos de matriz al sistema lineal
Tu notación es $\rho^{n,jm}_{n',jm'}=\langle n,jm|\rho|n',jm'\rangle$.
Ordenamos los estados con $n$ como índice externo y $m=-j,\ldots,j$ como índice interno:
$$q(n,m)=n(N+1)+(m+j).$$
La entrada matricial `rho[q(n,m), q(n_prime,m_prime)]` es tu elemento. Vectorizamos **por columnas**: $r_{q+dq'}=\rho_{q,q'}$.

Para construir los mismos coeficientes de las ecuaciones derivadas, usamos operadores dispersos y productos de Kronecker:
$$\mathcal L=-i(I\otimes H-H^T\otimes I)+\kappa\left[a^*\otimes a-\tfrac12 I\otimes a^\dagger a-\tfrac12(a^\dagger a)^T\otimes I\right].$$
Esto es la misma ecuación diferencial en forma matricial, sin una aproximación semiclasica. La aproximación numérica es el corte de fotones. Como $\mathcal L$ es constante, calculamos $\boldsymbol r(t)=e^{t\mathcal L}\boldsymbol r(0)$ con `expm_multiply`, sin formar la exponencial densa.


In [ ]:
def construir_operadores(N, n_max, omega, omega_0, lam):
    j = N / 2
    d_f = n_max + 1
    d_s = N + 1
    m_valores = np.arange(d_s, dtype=float) - j
    I_f = sparse.eye(d_f, format="csr", dtype=complex)
    I_s = sparse.eye(d_s, format="csr", dtype=complex)
    # a[n-1,n] = sqrt(n); J_+[m+1,m] = sqrt(j(j+1)-m(m+1)).
    a_f = sparse.diags(np.sqrt(np.arange(1, d_f)), offsets=1,
                       shape=(d_f, d_f), format="csr", dtype=complex)
    coef = np.sqrt(np.maximum(0, j*(j+1)-m_valores[:-1]*(m_valores[:-1]+1)))
    Jp_s = sparse.diags(coef, offsets=-1, shape=(d_s, d_s),
                        format="csr", dtype=complex)
    Jz_s = sparse.diags(m_valores, format="csr", dtype=complex)
    a = sparse.kron(a_f, I_s, format="csr")
    Jp = sparse.kron(I_f, Jp_s, format="csr")
    Jz = sparse.kron(I_f, Jz_s, format="csr")
    numero = (a.getH() @ a).tocsr()
    H = (omega*numero + omega_0*Jz
         + lam/np.sqrt(N) * ((a+a.getH()) @ (Jp+Jp.getH()))).tocsr()
    return H, a, numero, Jz, m_valores

def construir_liouvilliano(H, a, kappa):
    d = H.shape[0]
    I = sparse.eye(d, format="csr", dtype=complex)
    numero = (a.getH() @ a).tocsr()
    return (-1j*(sparse.kron(I, H)-sparse.kron(H.T, I))
            + kappa*(sparse.kron(a.conjugate(), a)
            - 0.5*sparse.kron(I, numero)
            - 0.5*sparse.kron(numero.T, I))).tocsr()

def indice_estado(n, m, N, n_max):
    j = N/2
    if not isinstance(n, (int, np.integer)) or not 0 <= n <= n_max:
        raise ValueError("n debe ser un entero entre 0 y n_max.")
    if (not np.isfinite(m) or abs(m)>j
            or not np.isclose(m+j, round(m+j), atol=1e-10, rtol=0)):
        raise ValueError("m debe pertenecer a -j, -j+1, ..., j.")
    return n*(N+1) + int(round(m+j))

def resolver_dicke(N, n_max, n_inicial, m_inicial,
                   omega, omega_0, lam, kappa, t_final, num_tiempos,
                   presupuesto_historia_gib=1.0):
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N debe ser un entero positivo.")
    if not isinstance(n_max, (int, np.integer)) or n_max < 0:
        raise ValueError("n_max debe ser un entero no negativo.")
    if not np.all(np.isfinite([omega, omega_0, lam, kappa, t_final])):
        raise ValueError("Los parámetros deben ser reales y finitos.")
    if kappa < 0 or t_final <= 0:
        raise ValueError("kappa >= 0 y t_final > 0.")
    if not isinstance(num_tiempos, (int, np.integer)) or num_tiempos < 2:
        raise ValueError("num_tiempos debe ser un entero >= 2.")
    q0 = indice_estado(n_inicial, m_inicial, N, n_max)
    d = (n_max+1)*(N+1)
    memoria_gib = num_tiempos*d*d*16 / 1024**3
    if not np.isfinite(presupuesto_historia_gib) or presupuesto_historia_gib <= 0:
        raise ValueError("presupuesto_historia_gib debe ser positivo y finito.")
    if memoria_gib > presupuesto_historia_gib:
        raise ValueError(f"Solo las matrices rho requieren {memoria_gib:.2f} GiB. "
                         "Reduce el corte o el número de tiempos para este cuaderno.")
    H, a, numero, Jz, m_valores = construir_operadores(N, n_max, omega, omega_0, lam)
    Liouv = construir_liouvilliano(H, a, kappa)
    r0 = np.zeros(d*d, dtype=complex)
    r0[q0+d*q0] = 1.0
    tiempos = np.linspace(0, t_final, num_tiempos)
    vectores = expm_multiply(Liouv, r0, start=0, stop=t_final,
                            num=num_tiempos, endpoint=True,
                            traceA=Liouv.diagonal().sum())
    # Cada fila de vectores está vectorizada por columnas.
    rho_t = vectores.reshape(num_tiempos, d, d).transpose(0, 2, 1)
    poblaciones = np.diagonal(rho_t, axis1=1, axis2=2).real
    poblaciones_nm = poblaciones.reshape(num_tiempos, n_max+1, N+1)
    P_n = poblaciones_nm.sum(axis=2)
    P_m = poblaciones_nm.sum(axis=1)
    n_medio = P_n @ np.arange(n_max+1)
    Jz_medio = P_m @ m_valores
    return dict(t=tiempos, rho=rho_t, P_n=P_n, P_m=P_m,
                n_medio=n_medio, Jz_medio=Jz_medio, m=m_valores,
                N=N, j=N/2, n_max=n_max, H=H, a=a, L=Liouv,
                n_inicial=n_inicial, m_inicial=m_inicial,
                parametros=dict(N=N, n_inicial=n_inicial, m_inicial=m_inicial,
                                omega=omega, omega_0=omega_0, lam=lam, kappa=kappa))


## Elegir el corte y resolver hasta convergencia
Se propone un corte inicial heurístico:
$$s=\frac{N\lambda^2}{\omega^2+(\kappa/2)^2},\qquad
n_{\max}^{(0)}=\max\big(12,\,2N,\,n_0+8,\,\lceil2s+8\rceil\big).$$
La escala $s$ se inspira en el desplazamiento del campo de una cavidad amortiguada; **no es una cota de fotones ni una fórmula exacta**. La dependencia de $m_0$, $\omega_0$ y el tiempo se incorpora al resolver la dinámica y comprobarla.

Cada cálculo se compara con el anterior, aumentando aproximadamente un 40% el corte. Se exige simultáneamente:
- poca probabilidad en los últimos tres niveles en todos los tiempos de salida;
- convergencia de $\langle n\rangle$ y $\langle J_z\rangle$;
- convergencia de las distribuciones $P_n$ y $P_m$, medidas por la distancia L1;
- convergencia de la matriz de densidad en hasta nueve tiempos, medida por la norma de Frobenius tras insertar ceros en los niveles adicionales.

Los criterios son diagnósticos numéricos en los tiempos muestreados. No se conservan dos historias completas de $\rho$ simultáneamente: entre cortes solo se retienen los observables y unas pocas matrices de referencia. Las gráficas usan exclusivamente el último cálculo, que haya superado los criterios.


In [ ]:
def resolver_dicke_automatico(N, n_inicial, m_inicial, omega, omega_0, lam,
                             kappa, t_final, num_tiempos,
                             tolerancia_borde=1e-7, atol_observables=1e-6,
                             rtol_observables=1e-5, tolerancia_distribuciones=1e-5,
                             tolerancia_rho=1e-5, niveles_borde=3,
                             max_iteraciones=8, presupuesto_memoria_gib=1.0):
    if omega <= 0 or not np.isfinite(omega):
        raise ValueError("Para esta estimación inicial, omega debe ser positiva.")
    if (not isinstance(max_iteraciones, (int, np.integer)) or max_iteraciones < 2
            or not isinstance(niveles_borde, (int, np.integer)) or niveles_borde < 1):
        raise ValueError("max_iteraciones >= 2 y niveles_borde >= 1, enteros.")
    tolerancias = [tolerancia_borde, atol_observables, rtol_observables,
                  tolerancia_distribuciones, tolerancia_rho, presupuesto_memoria_gib]
    if not np.all(np.isfinite(tolerancias)) or min(tolerancias) <= 0:
        raise ValueError("Las tolerancias y el presupuesto deben ser positivos y finitos.")
    if (not isinstance(N, (int, np.integer)) or N < 1
            or not isinstance(num_tiempos, (int, np.integer)) or num_tiempos < 2):
        raise ValueError("N >= 1 y num_tiempos >= 2, enteros.")
    if (not np.all(np.isfinite([lam, kappa, omega_0, t_final]))
            or kappa < 0 or t_final <= 0):
        raise ValueError("Parámetros finitos, kappa >= 0 y t_final > 0.")
    if not isinstance(n_inicial, (int, np.integer)) or n_inicial < 0:
        raise ValueError("n_inicial debe ser un entero no negativo.")
    escala = N*lam**2/(omega**2+(kappa/2)**2)
    corte = max(12, 2*N, n_inicial+8, int(np.ceil(2*escala+8)))
    # Reserva aproximada: historia, matriz dispersa y temporales.
    max_d_presupuesto = int(np.sqrt(presupuesto_memoria_gib*1024**3 /
                                   (16*(num_tiempos+100))))
    max_d = max_d_presupuesto
    corte_limite = max_d//(N+1)-1
    print(f"Estimación inicial n_max={corte}; límite estimado de recursos={corte_limite}.", flush=True)
    if corte > corte_limite:
        raise RuntimeError("El corte inicial supera el presupuesto. Reduce N o num_tiempos, "
                           "o aumenta presupuesto_memoria_gib si tu equipo dispone de RAM. "
                           "No se ha obtenido una solución convergida.")
    anterior = None
    historial = []
    for iteracion in range(max_iteraciones):
        print(f"Cálculo {iteracion+1}/{max_iteraciones}: n_max={corte}...", flush=True)
        actual = resolver_dicke(N, corte, n_inicial, m_inicial, omega, omega_0,
                               lam, kappa, t_final, num_tiempos,
                               presupuesto_historia_gib=presupuesto_memoria_gib)
        idx = np.unique(np.linspace(0, num_tiempos-1, min(9,num_tiempos), dtype=int))
        borde = float(actual["P_n"][:, -min(niveles_borde,corte+1):].sum(axis=1).max())
        fila = dict(n_max=corte, borde=borde, error_n=None, error_Jz=None,
                    error_Pn=None, error_Pm=None, error_rho=None, convergido=False)
        if anterior is not None:
            error_n = float(np.max(np.abs(actual["n_medio"]-anterior["n_medio"])))
            error_Jz = float(np.max(np.abs(actual["Jz_medio"]-anterior["Jz_medio"])))
            Pn_anterior = np.pad(anterior["P_n"], ((0,0),(0,corte-anterior["n_max"])))
            error_Pn = float(np.abs(actual["P_n"]-Pn_anterior).sum(axis=1).max())
            error_Pm = float(np.abs(actual["P_m"]-anterior["P_m"]).sum(axis=1).max())
            # Los estados anteriores ocupan el bloque superior izquierdo,
            # porque el orden de la base conserva m como índice interno.
            d_old = anterior["rho_muestra"].shape[1]
            error_rho = 0.0
            for pos, it in enumerate(idx):
                delta = actual["rho"][it].copy()
                delta[:d_old,:d_old] -= anterior["rho_muestra"][pos]
                error_rho = max(error_rho, float(np.linalg.norm(delta, ord="fro")))
            del delta, Pn_anterior
            observables_ok = (np.allclose(actual["n_medio"], anterior["n_medio"],
                                           atol=atol_observables, rtol=rtol_observables)
                              and np.allclose(actual["Jz_medio"], anterior["Jz_medio"],
                                              atol=atol_observables, rtol=rtol_observables))
            convergido = (borde <= tolerancia_borde and observables_ok
                          and max(error_Pn,error_Pm) <= tolerancia_distribuciones
                          and error_rho <= tolerancia_rho)
            fila.update(error_n=error_n, error_Jz=error_Jz, error_Pn=error_Pn,
                        error_Pm=error_Pm, error_rho=error_rho, convergido=convergido)
            print(f"  borde={borde:.2e}, delta<n>={error_n:.2e}, "
                  f"delta<Jz>={error_Jz:.2e}, delta rho={error_rho:.2e}", flush=True)
            if convergido:
                historial.append(fila)
                actual["historial_cortes"] = historial
                actual["convergido"] = True
                print(f"Convergencia alcanzada: n_max={corte}.", flush=True)
                return actual
        else:
            print(f"  borde={borde:.2e}; falta comparar con un corte mayor.", flush=True)
        historial.append(fila)
        nuevo_corte = max(corte+4, int(np.ceil(1.4*(corte+1)))-1)
        nuevo_corte = min(nuevo_corte, corte_limite)
        if nuevo_corte <= corte or iteracion+1 == max_iteraciones:
            motivo = "presupuesto de recursos" if nuevo_corte <= corte else "máximo de iteraciones"
            del actual
            gc.collect()
            raise RuntimeError(f"No se alcanzó convergencia: {motivo}. Último corte={corte}, "
                               f"ocupación del borde={borde:.2e}. Ajusta los recursos o "
                               "max_iteraciones; no se aceptó el resultado como convergido.")
        anterior = {clave: actual[clave].copy() for clave in
                    ["n_medio", "Jz_medio", "P_n", "P_m"]}
        anterior["n_max"] = corte
        anterior["rho_muestra"] = actual["rho"][idx].copy()
        del actual
        gc.collect()
        corte = nuevo_corte

# Evita usar o retener un resultado viejo al repetir esta celda.
for nombre in ("resultado_cuantico", "rho_t", "valores_elemento", "resultado_mayor", "t"):
    globals().pop(nombre, None)
gc.collect()
resultado_cuantico = resolver_dicke_automatico(
    N, n_inicial, m_inicial, omega, omega_0, lam, kappa, t_final, num_tiempos,
    tolerancia_borde, atol_observables, rtol_observables,
    tolerancia_distribuciones, tolerancia_rho, niveles_borde,
    max_iteraciones, presupuesto_memoria_gib)
n_max = resultado_cuantico["n_max"]
t = resultado_cuantico["t"]
rho_t = resultado_cuantico["rho"]
print(f"Resultado final: n_max={n_max}, d={rho_t.shape[1]}, {len(t)} tiempos.")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
ax[0].plot(t, resultado_cuantico["n_medio"])
ax[0].set(xlabel="Tiempo", ylabel=r"$\langle a^\dagger a\rangle$", title="Número medio de fotones")
ax[1].plot(t, resultado_cuantico["Jz_medio"], color="darkorange")
ax[1].set(xlabel="Tiempo", ylabel=r"$\langle J_z\rangle$", title="Inversión colectiva")
for eje in ax:
    eje.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
im = ax[0].pcolormesh(t, np.arange(n_max+1), resultado_cuantico["P_n"].T,
                      shading="auto", cmap="viridis")
fig.colorbar(im, ax=ax[0], label="Probabilidad")
ax[0].set(xlabel="Tiempo", ylabel="n", title="Distribución de fotones")
im = ax[1].pcolormesh(t, resultado_cuantico["m"], resultado_cuantico["P_m"].T,
                      shading="auto", cmap="viridis")
fig.colorbar(im, ax=ax[1], label="Probabilidad")
ax[1].set(xlabel="Tiempo", ylabel="m", title="Distribución del espín")
plt.show()


## Consultar cualquier elemento usando tus índices
`elemento_matriz(resultado_cuantico, n, m, n_prima, m_prima)` devuelve la evolución de $\rho^{n,jm}_{n',jm'}$. Si ambos pares coinciden, es una población; en otro caso es una coherencia.


In [ ]:
def elemento_matriz(resultado_cuantico, n, m, n_prima, m_prima):
    q = indice_estado(n, m, resultado_cuantico["N"], resultado_cuantico["n_max"])
    qp = indice_estado(n_prima, m_prima, resultado_cuantico["N"], resultado_cuantico["n_max"])
    return resultado_cuantico["rho"][:, q, qp]

def graficar_elemento(resultado_cuantico, n, m, n_prima, m_prima):
    valores = elemento_matriz(resultado_cuantico, n, m, n_prima, m_prima)
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.plot(resultado_cuantico["t"], valores.real, label="Parte real")
    ax.plot(resultado_cuantico["t"], valores.imag, "--", label="Parte imaginaria")
    ax.set(xlabel="Tiempo", ylabel="Elemento de rho",
           title=f"rho: bra (n={n}, m={m:g}); ket (n'={n_prima}, m'={m_prima:g}); j={resultado_cuantico['j']:g}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()
    return valores

# Ejemplo: población del estado inicial. Cambia los cuatro índices libremente.
valores_elemento = graficar_elemento(resultado_cuantico, n_inicial, m_inicial,
                                   n_inicial, m_inicial)
# Ejemplo de coherencia, si n_max >= 1:
# valores_elemento = graficar_elemento(resultado_cuantico, 0, -j, 1, -j+1)


## Comprobaciones físicas y corte elegido
La traza debe ser uno y la matriz debe permanecer hermítica y positiva dentro del error numérico. Se inspecciona la positividad en hasta nueve tiempos si $d\leq250$.

El ajuste automático ya comparó cortes, distribuciones, observables y matrices muestreadas. Esta celda muestra además la ocupación del último nivel y de la banda de borde elegida.


In [ ]:
trazas = np.trace(rho_t, axis1=1, axis2=2)
error_traza = np.max(np.abs(trazas-1))
indices_revision = np.unique(np.linspace(0, len(t)-1, min(9, len(t)), dtype=int))
error_hermiticidad = max(np.max(np.abs(rho_t[k]-rho_t[k].conj().T))
                        for k in indices_revision)
p_borde = resultado_cuantico["P_n"][:, -1]
print(f"Máximo error de traza: {error_traza:.3e}")
print(f"Error de hermiticidad en tiempos revisados: {error_hermiticidad:.3e}")
if rho_t.shape[1] <= 250:
    menor_autovalor = min(np.linalg.eigvalsh((rho_t[k]+rho_t[k].conj().T)/2).min()
                         for k in indices_revision)
    print(f"Menor autovalor en tiempos revisados: {menor_autovalor:.3e}")
else:
    print("Positividad no inspeccionada: d > 250.")
print(f"Máxima probabilidad en n_max: {p_borde.max():.3e}")
print(f"Máxima probabilidad en banda de borde: {resultado_cuantico['P_n'][:, -min(niveles_borde,n_max+1):].sum(axis=1).max():.3e}")
if p_borde.max() > 1e-4:
    print("Aumenta n_max y compara los observables antes de aceptar el resultado_cuantico.")
plt.figure(figsize=(7, 3))
plt.plot(t, p_borde)
plt.xlabel("Tiempo")
plt.ylabel(r"$P(n_{\max})$")
plt.title("Ocupación del borde del espacio de fotones")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Historial de los cortes calculados automáticamente.
print(f"{'n_max':>7} {'P_borde':>12} {'delta <n>':>12} {'delta <Jz>':>12} {'delta rho':>12} {'aceptado':>10}")
for fila in resultado_cuantico["historial_cortes"]:
    def formato(valor):
        return "--" if valor is None else f"{valor:.2e}"
    print(f"{fila['n_max']:7d} {fila['borde']:12.2e} "
          f"{formato(fila['error_n']):>12} {formato(fila['error_Jz']):>12} "
          f"{formato(fila['error_rho']):>12} {str(fila['convergido']):>10}")


## Control con solución conocida
Esta celda hace un cálculo independiente sin interacción ($\lambda=0$), con tres fotones iniciales. En ese caso la solución conocida es $\langle n\rangle=3e^{-\kappa t}$ y $\langle J_z\rangle=m_{\mathrm{inicial}}$. No modifica el resultado principal.


In [ ]:
control = resolver_dicke(1, 3, 3, -0.5, 1.0, 1.0, 0.0, 0.2, 10.0, 101)
exacta = 3*np.exp(-0.2*control["t"])
print("Error frente a la solución conocida:", np.max(np.abs(control["n_medio"]-exacta)))
assert np.allclose(control["n_medio"], exacta, atol=1e-10, rtol=1e-10)
assert np.allclose(control["Jz_medio"], -0.5, atol=1e-10)
print("Control superado.")


## Valor esperado de la energía del cálculo cuántico

Calculamos

$$E(t)=\operatorname{Tr}[\rho(t)H]
=\sum_{i,j}\rho_{ij}(t)H_{ji}.$$

El Hamiltoniano incluye campo, espín e interacción. Las pérdidas afectan
rho(t), y por eso afectan la energía media aunque kappa no esté en H.
Usamos el Hamiltoniano guardado junto con la misma solución cuántica.


In [ ]:
H_cuantico = resultado_cuantico["H"].toarray()
energia_compleja = np.einsum(
    "tij,ji->t", resultado_cuantico["rho"], H_cuantico, optimize=True
)
energia_media = energia_compleja.real
resultado_cuantico["energia_media"] = energia_media
print("Máxima parte imaginaria de la energía:", np.max(np.abs(energia_compleja.imag)))
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.plot(resultado_cuantico["t"], energia_media)
ax.set(xlabel="Tiempo", ylabel=r"$\langle H\rangle(t)$", title="Energía media cuántica")
ax.grid(alpha=0.3)
plt.show()


## Función para buscar un estado aproximadamente estacionario

Un estado estacionario satisface la ecuación

$$
\mathcal{L}[\rho]=0.
$$

Para comprobarlo numéricamente, calculamos la tasa relativa de cambio

$$
\epsilon(t)=
\frac{\|\mathcal{L}[\rho(t)]\|_F}{\|\rho(t)\|_F},
$$

donde la norma de Frobenius es

$$
\|A\|_F=\sqrt{\sum_{i,j}|A_{ij}|^2}.
$$

La función resuelve primero la evolución con un corte de fotones n_max
determinado automáticamente.

Después busca el primer tiempo muestreado para el cual epsilon(t)
permanece por debajo de tol_tasa en todos los tiempos posteriores
calculados. También exige que queden al menos tiempo_confirmacion
unidades de tiempo para confirmar esa condición.

El resultado es un tiempo estacionario aproximado y la matriz
rho(t_estacionario). La precisión temporal depende de la separación
entre los tiempos muestreados.

Si no se cumple el criterio dentro del intervalo, la función informa
que no encontró estacionariedad. Esta celda únicamente define la
función; los parámetros y su ejecución aparecen en la siguiente sección.

Si no se confirma estacionariedad, la función devuelve `None` para el
tiempo y la matriz estacionaria, junto con la evolución calculada.
Esto permite examinar el diagnóstico y aumentar el intervalo sin
presentar el último estado como estacionario.


In [ ]:
def hallar_rho_estacionaria(
    N, n_inicial, m_inicial, omega, omega_0, lam, kappa,
    t_max=200.0, num_tiempos=401,
    tol_tasa=1e-4, tiempo_confirmacion=10.0,
    **opciones_corte
):
    if not np.isfinite(tol_tasa) or tol_tasa <= 0:
        raise ValueError("tol_tasa debe ser positiva y finita.")

    if (
        not np.isfinite(t_max)
        or not np.isfinite(tiempo_confirmacion)
        or not 0 < tiempo_confirmacion < t_max
    ):
        raise ValueError(
            "Se requieren valores finitos y "
            "0 < tiempo_confirmacion < t_max."
        )

    # Calcula rho(t) y determina n_max automáticamente.
    solucion = resolver_dicke_automatico(
        N=N,
        n_inicial=n_inicial,
        m_inicial=m_inicial,
        omega=omega,
        omega_0=omega_0,
        lam=lam,
        kappa=kappa,
        t_final=t_max,
        num_tiempos=num_tiempos,
        **opciones_corte
    )

    tiempos = solucion["t"]
    historia_rho = solucion["rho"]
    Liouv = solucion["L"]

    tasas = np.empty(len(tiempos))

    for i, rho in enumerate(historia_rho):
        # Misma vectorización por columnas que usa el solver.
        vector_rho = rho.reshape(-1, order="F")
        vector_derivada = Liouv @ vector_rho

        tasas[i] = (
            np.linalg.norm(vector_derivada)
            / np.linalg.norm(vector_rho)
        )

    if not np.all(np.isfinite(tasas)):
        raise RuntimeError(
            "Se obtuvieron tasas no finitas; revisa la solución numérica."
        )

    # Máxima tasa desde cada instante hasta el final del cálculo.
    maxima_tasa_posterior = np.maximum.accumulate(tasas[::-1])[::-1]

    candidatos = np.flatnonzero(
        (maxima_tasa_posterior <= tol_tasa)
        & ((tiempos[-1] - tiempos) >= tiempo_confirmacion)
    )

    solucion["tasa_relativa"] = tasas
    solucion["tol_tasa"] = tol_tasa
    solucion["estacionario_confirmado"] = candidatos.size > 0
    if candidatos.size == 0:
        solucion["indice_estacionario"] = None
        return None, None, solucion

    indice = int(candidatos[0])
    t_estacionario = float(tiempos[indice])
    rho_estacionaria = historia_rho[indice].copy()

    solucion["tasa_relativa"] = tasas
    solucion["indice_estacionario"] = indice
    solucion["tol_tasa"] = tol_tasa

    return t_estacionario, rho_estacionaria, solucion


## Búsqueda con los parámetros elegidos

Elegimos los parámetros físicos, el intervalo de búsqueda y la
tolerancia de estacionariedad.

El diccionario parametros toma inicialmente los valores definidos
anteriormente en el notebook. Se pueden sustituir por otros valores;
m_inicial debe pertenecer a -N/2, -N/2+1, ..., N/2.

El corte n_max se determina automáticamente para estos parámetros y
para el intervalo temporal solicitado.

Usamos inicialmente tol_tasa = 1e-4. Esta tolerancia tiene unidades
de inverso de tiempo y limita la tasa relativa de cambio de rho.
No representa directamente la distancia al estado estacionario exacto.

La búsqueda devuelve t_estacionario y rho_estacionaria. La gráfica
muestra la tasa de cambio, la tolerancia elegida y el tiempo encontrado.

La estabilización visual de algunos observables puede ocurrir antes
de que la matriz de densidad completa cumpla este criterio.

El punto de la trayectoria aceptado es una aproximación con tasa de cambio
pequeña, no una solución algebraica exacta del núcleo del Liouvilliano.


In [ ]:
# Parámetros de la búsqueda cuántica; independientes de campo medio.
parametros = parametros_cuanticos.copy()
busqueda = {
    "t_max": 200.0, "num_tiempos": 401,
    "tol_tasa": 1e-4, "tiempo_confirmacion": 10.0,
}
opciones_corte = {"presupuesto_memoria_gib": 1.0, "max_iteraciones": 8}

t_estacionario = rho_estacionaria = solucion_estacionaria = None
t_estacionario, rho_estacionaria, solucion_estacionaria = hallar_rho_estacionaria(
    **parametros, **busqueda, **opciones_corte
)
tasas_estacionarias = solucion_estacionaria["tasa_relativa"]
print("Parámetros cuánticos:", parametros)
print("n_max:", solucion_estacionaria["n_max"])
if rho_estacionaria is None:
    print("NO se confirmó estacionariedad en el intervalo calculado.")
    print(f"Tasa final = {tasas_estacionarias[-1]:.3e}. Amplía t_max si es necesario.")
else:
    print(f"Tiempo estacionario aproximado: {t_estacionario:g}")
    print("Traza:", np.trace(rho_estacionaria))

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.semilogy(solucion_estacionaria["t"], np.maximum(tasas_estacionarias, 1e-300))
ax.axhline(busqueda["tol_tasa"], color="red", linestyle="--", label="Tolerancia")
if t_estacionario is not None:
    ax.axvline(t_estacionario, color="green", linestyle=":", label="Tiempo aceptado")
ax.set(xlabel="Tiempo", ylabel=r"$\|\mathcal{L}[\rho]\|_F/\|\rho\|_F$",
       title="Diagnóstico de estacionariedad cuántica")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## Funciones de campo medio: puntos estacionarios y modos

Estas funciones utilizan las mismas ecuaciones de campo medio de la
sección de evolución. `longitud_espin` fija
$r=\sqrt{x^2+y^2+z^2}$, conservado por estas ecuaciones; por defecto $r=1/2$.
Para comparar con una trayectoria utiliza su longitud inicial.

Con $D=\omega^2+(\kappa/2)^2$:

$$\lambda_c=\sqrt{\frac{\omega_0D}{8r\omega}}.$$

El punto normal inferior es $(0,0,0,0,-r)$. Encima del umbral,

$$\mu=(\lambda_c/\lambda)^2,\quad
x=\pm r\sqrt{1-\mu^2},\quad y=0,\quad z=-r\mu,$$
$$u=-2\lambda\omega x/D,\qquad v=\kappa u/(2\omega).$$

La función devuelve también el normal invertido. `fase="auto"` elige
el normal inferior hasta el crítico y una rama superradiante encima.
Con kappa=0 encuentra puntos fijos algebraicos, no garantiza relajación.

El Jacobiano se restringe a perturbaciones tangentes a la esfera del espín.
Sus polos $s=-\gamma+i\Omega$ dan frecuencias angulares y decaimientos
de amplitud. LP/UP se ordenan localmente por frecuencia cuando son distintas.

**Frecuencias degeneradas:** las dos frecuencias comunes se conservan,
pero gamma_LP y gamma_UP quedan como NaN; no se eligen por redondeo.
Las tasas de los modos sin etiqueta se guardan en `gamma_degeneradas`
y en `seleccionado["modos"]`. Esto no establece un seguimiento global
de ramas a través de cruces o puntos excepcionales.

**Modos no oscilatorios:** Omega=0 y se conservan sus dos polos reales,
con sus dos decaimientos. No se inventa un único gamma. Si los cuatro
polos son reales, se muestran huecos en las curvas LP/UP.

No hay corte n_max en campo medio. A lambda colectivo y r fijos, N escala
los valores medios, pero no cambia estas frecuencias. Esto no implica
independencia de N en el sistema cuántico finito.


In [ ]:
"""Funciones independientes para el notebook corregido de Dicke abierto."""
import itertools
import numpy as np


def _rhs_mf_punto(q, omega, omega_0, lam, kappa):
    u, v, x, y, z = q
    return np.array([
        -kappa*u/2 + omega*v,
        -omega*u - kappa*v/2 - 2*lam*x,
        -omega_0*y,
        omega_0*x - 4*lam*u*z,
        4*lam*u*y,
    ])


def analizar_punto_campo_medio(q, omega, omega_0, lam, kappa):
    """Analiza fluctuaciones a longitud de espín fija; no inventa etiquetas
    de decaimiento LP/UP cuando las frecuencias son degeneradas.
    """
    q = np.asarray(q, dtype=float)
    if q.shape != (5,) or not np.all(np.isfinite(q)):
        raise ValueError("El punto debe contener cinco valores finitos.")
    u, _, x, y, z = q
    espin = q[2:]
    if np.linalg.norm(espin) < 1e-12:
        raise ValueError("La reducción tangente requiere espín medio no nulo.")

    jacobiano = np.array([
        [-kappa/2, omega, 0, 0, 0],
        [-omega, -kappa/2, -2*lam, 0, 0],
        [0, 0, 0, -omega_0, 0],
        [-4*lam*z, 0, omega_0, 0, -4*lam*u],
        [4*lam*y, 0, 0, 4*lam*u, 0],
    ])
    # SVD: dos direcciones ortonormales perpendiculares al espín.
    _, _, vh = np.linalg.svd(espin.reshape(1, 3), full_matrices=True)
    base = np.zeros((5, 4))
    base[:2, :2] = np.eye(2)
    base[2:, 2:] = vh[1:].T
    matriz = base.T @ jacobiano @ base
    valores, vectores = np.linalg.eig(matriz)
    tol = 1e-8 * max(1.0, np.max(np.abs(valores)))
    if np.any(valores.real > tol):
        estabilidad = "inestable"
    elif np.all(valores.real < -tol):
        estabilidad = "estable"
    else:
        estabilidad = "marginal"

    positivos = np.flatnonzero(valores.imag > tol)
    negativos = np.flatnonzero(valores.imag < -tol)
    pares = []
    if len(positivos) != len(negativos):
        raise RuntimeError("No se pudieron agrupar los polos conjugados.")
    if len(positivos):
        # Asignación conjunta: cada polo negativo se utiliza una sola vez.
        emparejamiento = min(
            itertools.permutations(negativos),
            key=lambda perm: sum(abs(valores[i].conjugate()-valores[j])
                                 for i, j in zip(positivos, perm)),
        )
        pares.extend(valores[[i, j]] for i, j in zip(positivos, emparejamiento))
    reales = valores[np.abs(valores.imag) <= tol]
    if len(reales) == 2:
        pares.append(reales[np.argsort(reales.real)])

    modos = []
    for par in pares:
        oscilatorio = bool(np.max(np.abs(par.imag)) > tol)
        gammas = np.sort(-par.real)
        gammas[np.abs(gammas) <= tol] = 0.0
        modos.append({
            "omega": float(np.max(np.abs(par.imag))) if oscilatorio else 0.0,
            "gamma": float(gammas.mean()) if oscilatorio else np.nan,
            "gammas": gammas,
            "oscilatorio": oscilatorio,
            "autovalores": par,
            "ambigua": False,
        })

    ramas = {}
    degenerada = False
    if len(modos) == 2:
        modos.sort(key=lambda modo: modo["omega"])
        tol_degeneracion = 1e-7 * max(1.0, modos[1]["omega"])
        degenerada = abs(modos[1]["omega"]-modos[0]["omega"]) <= tol_degeneracion
        if degenerada:
            # Frecuencia común válida; el nombre LP/UP no determina un gamma.
            frecuencia_comun = float(np.mean([m["omega"] for m in modos]))
            for nombre in ("LP", "UP"):
                ramas[nombre] = {
                    "omega": frecuencia_comun, "gamma": np.nan,
                    "gammas": np.full(2, np.nan),
                    "oscilatorio": modos[0]["oscilatorio"],
                    "ambigua": True,
                }
        else:
            ramas = dict(zip(("LP", "UP"), modos))

    return {
        "estado": q.copy(),
        "residuo": float(np.linalg.norm(_rhs_mf_punto(q, omega, omega_0, lam, kappa))),
        "jacobiano": jacobiano,
        "base": base,
        "matriz_reducida": matriz,
        "autovalores": valores,
        "autovectores": base @ vectores,
        "estabilidad": estabilidad,
        "modos": modos,
        "ramas": ramas,
        "frecuencia_degenerada": degenerada,
        "asignacion_disponible": len(modos) == 2,
    }


def puntos_y_modos_campo_medio(lam, omega=1.0, omega_0=1.0,
                              kappa=0.1, N=20, fase="auto", signo=1,
                              longitud_espin=0.5):
    """Puntos fijos con longitud de espín conservada r=longitud_espin.

    'auto' selecciona normal inferior o superradiante según el umbral.
    En degeneraciones, consulta 'modos': gamma_LP/UP queda indefinido.
    Con kappa=0 se calculan puntos fijos, no se asegura relajación.
    """
    if not np.all(np.isfinite([lam, omega, omega_0, kappa, longitud_espin])):
        raise ValueError("Los parámetros deben ser finitos.")
    if omega <= 0 or omega_0 <= 0 or lam < 0 or kappa < 0:
        raise ValueError("Frecuencias positivas; lambda y kappa >= 0.")
    if not 0 < longitud_espin <= 0.5:
        raise ValueError("Se requiere 0 < longitud_espin <= 1/2.")
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N debe ser un entero positivo.")
    if fase not in ("auto", "normal", "superradiante") or signo not in (-1, 1):
        raise ValueError("fase: auto/normal/superradiante; signo: +1/-1.")

    r = longitud_espin
    D = omega**2 + (kappa/2)**2
    critica = np.sqrt(omega_0*D/(8*r*omega))
    estados = {
        "normal": np.array([0., 0., 0., 0., -r]),
        "normal_invertido": np.array([0., 0., 0., 0., r]),
    }
    if lam > critica:
        mu = (critica/lam)**2
        for s in (-1, 1):
            x = s*r*np.sqrt(max(0., 1-mu**2))
            u = -2*lam*omega*x/D
            estados[f"superradiante_{s:+d}"] = np.array([
                u, kappa*u/(2*omega), x, 0., -r*mu,
            ])
    if fase == "superradiante" and lam <= critica:
        raise ValueError("La rama superradiante no nula exige lambda > lambda_c.")
    elegida = "normal"
    if fase == "superradiante" or (fase == "auto" and lam > critica):
        elegida = f"superradiante_{signo:+d}"
    puntos = {}
    for nombre, q in estados.items():
        info = analizar_punto_campo_medio(q, omega, omega_0, lam, kappa)
        info.update({"a_medio": np.sqrt(N)*(q[0]+1j*q[1]),
                     "espin_medio": N*q[2:]})
        puntos[nombre] = info
    return {"lambda": lam, "kappa": kappa, "lambda_c": critica,
            "longitud_espin": r, "puntos": puntos,
            "rama_elegida": elegida, "seleccionado": puntos[elegida]}


def _barrer_parametro_mf(valores, parametro, parametros):
    valores = np.asarray(valores, dtype=float)
    if valores.ndim != 1 or valores.size == 0 or not np.all(np.isfinite(valores)):
        raise ValueError("El barrido requiere un vector finito no vacío.")
    resultados = []
    for valor in valores:
        argumentos = dict(parametros)
        argumentos[parametro] = float(valor)
        resultados.append(puntos_y_modos_campo_medio(**argumentos))
    curvas = {"lambda" if parametro == "lam" else "kappa": valores,
              "lambda_c": np.array([r["lambda_c"] for r in resultados]),
              "resultados": resultados,
              "degenerada": np.array([r["seleccionado"]["frecuencia_degenerada"]
                                      for r in resultados]),
              "gamma_degeneradas": np.full((valores.size, 2), np.nan)}
    for nombre in ("LP", "UP"):
        curvas[f"omega_{nombre}"] = np.full(valores.size, np.nan)
        curvas[f"gamma_{nombre}"] = np.full(valores.size, np.nan)
        curvas[f"gammas_{nombre}"] = np.full((valores.size, 2), np.nan)
        curvas[f"no_oscilatoria_{nombre}"] = np.zeros(valores.size, dtype=bool)
    for i, resultado in enumerate(resultados):
        info = resultado["seleccionado"]
        if info["frecuencia_degenerada"]:
            curvas["gamma_degeneradas"][i] = sorted(m["gamma"] for m in info["modos"])
        for nombre, modo in info["ramas"].items():
            curvas[f"omega_{nombre}"][i] = modo["omega"]
            curvas[f"gamma_{nombre}"][i] = modo["gamma"]
            curvas[f"gammas_{nombre}"][i] = modo["gammas"]
            curvas[f"no_oscilatoria_{nombre}"][i] = not modo["oscilatorio"]
    return curvas


def barrer_lambda_campo_medio(lambdas, **parametros):
    curvas = _barrer_parametro_mf(lambdas, "lam", parametros)
    curvas["lambda_c"] = float(curvas["lambda_c"][0])
    return curvas


def barrer_kappa_campo_medio(kappas, lam, **parametros):
    parametros = dict(parametros, lam=lam)
    curvas = _barrer_parametro_mf(kappas, "kappa", parametros)
    omega = parametros.get("omega", 1.0)
    omega_0 = parametros.get("omega_0", 1.0)
    r = parametros.get("longitud_espin", 0.5)
    argumento = 8*r*lam**2*omega/omega_0 - omega**2
    curvas["kappa_c"] = 2*np.sqrt(argumento) if argumento > 0 else None
    curvas["lambda_fija"] = lam
    return curvas


def malla_con_critico(minimo, maximo, critico=None, num=501):
    if not np.all(np.isfinite([minimo, maximo])) or not 0 <= minimo < maximo:
        raise ValueError("Se requiere 0 <= minimo < maximo finitos.")
    valores = np.linspace(minimo, maximo, num)
    if critico is not None and minimo <= critico <= maximo:
        cerca = critico*np.linspace(0.98, 1.02, 161)
        cerca = cerca[(cerca >= minimo) & (cerca <= maximo)]
        valores = np.unique(np.r_[valores, cerca, critico])
    return valores


def avisos_barrido_mf(curvas):
    if np.any(curvas["degenerada"]):
        print("Frecuencias coincidentes: gamma_LP/UP queda sin asignar. "
              "Las dos tasas están en gamma_degeneradas y en los modos completos.")
    if any(np.any(~np.isfinite(curvas[f"omega_{n}"])) for n in ("LP", "UP")):
        print("Hay puntos sin clasificación local LP/UP; consulta los autovalores completos.")


## Evolución y estado estacionario en campo medio

Definimos los valores medios normalizados

$$
\alpha=\frac{\langle a\rangle}{\sqrt N}=u+iv,\qquad
x=\frac{\langle J_x\rangle}{N},\quad
y=\frac{\langle J_y\rangle}{N},\quad
z=\frac{\langle J_z\rangle}{N}.
$$

Factorizamos las correlaciones entre el campo y el espín colectivo.
Con pérdida de fotones descrita por el operador de salto sqrt(kappa) a,
las ecuaciones son

$$
\begin{aligned}
\dot u &= -\frac{\kappa}{2}u+\omega v,\\
\dot v &= -\omega u-\frac{\kappa}{2}v-2\lambda x,\\
\dot x &= -\omega_0y,\\
\dot y &= \omega_0x-4\lambda uz,\\
\dot z &= 4\lambda uy.
\end{aligned}
$$

Las resolvemos para los parámetros y condiciones iniciales especificados.
No necesitamos un corte de fotones n_max en esta representación.

El ejemplo comienza con los átomos en el estado superior (z = +1/2) y una pequeña
amplitud inicial de la cavidad. Esta perturbación permite salir del
punto normal si es inestable.

Para el punto normal con z = -1/2, el umbral de campo medio es

$$
\lambda_c=\frac12
\sqrt{\frac{\omega_0}{\omega}
\left[\omega^2+\left(\frac{\kappa}{2}\right)^2\right]}.
$$

El ejemplo usa lambda = 0.5 y kappa = 0.1, por debajo del umbral
lambda_c = 0.50062461 para las frecuencias elegidas. Se puede cambiar para explorar otros regímenes.

Buscamos estacionariedad mediante

$$
\epsilon(t)=\frac{\|\dot{\mathbf q}(t)\|_2}
{\|\mathbf q(t)\|_2},
\qquad
\mathbf q=(u,v,x,y,z).
$$

Aceptamos el primer tiempo muestreado a partir del cual la tasa permanece
bajo la tolerancia hasta el final del cálculo, siempre que quede un
intervalo suficiente de confirmación.

El resultado es un punto estacionario de campo medio, no una matriz
de densidad cuántica completa. La estimación del número de fotones
en esta aproximación es N |alpha|^2 y no incluye las fluctuaciones
cuánticas del campo.

La configuración de esta evolución es independiente de la cuántica.
Con longitud inicial r distinta de 1/2, el umbral correspondiente se
calcula con ese r. La conservación de la longitud se comprueba al final.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


def ecuaciones_campo_medio(t, estado, omega, omega_0, lam, kappa):
    return _rhs_mf_punto(estado, omega, omega_0, lam, kappa)


# Parámetros físicos: modifica estos valores.
parametros_mf = {
    "N": 20,
    "omega": 1.0,
    "omega_0": 1.0,
    "lam": 0.50,
    "kappa": 0.10,
}

# Condiciones iniciales normalizadas.
# alpha = <a>/sqrt(N); x,y,z = <Jx,Jy,Jz>/N.
inicial_mf = {
    "alpha": 1e-4 + 0j,
    "x": 0.0,
    "y": 0.0,
    "z": 0.5,
}

# Configuración temporal y criterio de estacionariedad.
t_max_mf = 900
num_tiempos_mf = 601
tol_tasa_mf = 1e-7
tiempo_confirmacion_mf = 20.0

N_mf = parametros_mf["N"]
omega_mf = parametros_mf["omega"]
omega_0_mf = parametros_mf["omega_0"]
lam_mf = parametros_mf["lam"]
kappa_mf = parametros_mf["kappa"]

if not isinstance(N_mf, (int, np.integer)) or N_mf < 1:
    raise ValueError("N debe ser un entero positivo.")

if not np.all(np.isfinite([omega_mf, omega_0_mf, lam_mf, kappa_mf])):
    raise ValueError("Los parámetros físicos deben ser finitos.")

if omega_mf <= 0 or omega_0_mf <= 0 or kappa_mf < 0:
    raise ValueError("Se requiere omega > 0, omega_0 > 0 y kappa >= 0.")

if not 0 < tiempo_confirmacion_mf < t_max_mf:
    raise ValueError("Se requiere 0 < tiempo_confirmacion < t_max.")

if not np.isfinite(tol_tasa_mf) or tol_tasa_mf <= 0:
    raise ValueError("La tolerancia debe ser positiva y finita.")

alpha_inicial_mf = complex(inicial_mf["alpha"])
q0_mf = np.array([
    alpha_inicial_mf.real,
    alpha_inicial_mf.imag,
    inicial_mf["x"],
    inicial_mf["y"],
    inicial_mf["z"],
], dtype=float)

if not np.all(np.isfinite(q0_mf)):
    raise ValueError("Las condiciones iniciales deben ser finitas.")

if np.linalg.norm(q0_mf[2:]) > 0.5 + 1e-12:
    raise ValueError("El espín normalizado debe satisfacer x²+y²+z² <= 1/4.")

args_mf = (omega_mf, omega_0_mf, lam_mf, kappa_mf)
tiempos_mf = np.linspace(0, t_max_mf, num_tiempos_mf)

integracion_mf = solve_ivp(
    ecuaciones_campo_medio,
    (0, t_max_mf),
    q0_mf,
    args=args_mf,
    t_eval=tiempos_mf,
    method="DOP853",
    rtol=1e-9,
    atol=1e-11,
)

if not integracion_mf.success:
    raise RuntimeError(integracion_mf.message)

historia_mf = integracion_mf.y.T
u_mf, v_mf, x_mf, y_mf, z_mf = historia_mf.T
alpha_mf = u_mf + 1j * v_mf
fotones_mf = N_mf * np.abs(alpha_mf)**2

derivadas_mf = np.array([
    ecuaciones_campo_medio(t, q, *args_mf)
    for t, q in zip(tiempos_mf, historia_mf)
])

tasas_mf = (
    np.linalg.norm(derivadas_mf, axis=1)
    / np.maximum(np.linalg.norm(historia_mf, axis=1), 1e-15)
)

maxima_tasa_posterior_mf = np.maximum.accumulate(tasas_mf[::-1])[::-1]
candidatos_mf = np.flatnonzero(
    (maxima_tasa_posterior_mf <= tol_tasa_mf)
    & ((tiempos_mf[-1] - tiempos_mf) >= tiempo_confirmacion_mf)
)

# Reinicia los resultados estacionarios en cada ejecución.
t_estacionario_mf = None
estado_estacionario_mf = None

resultado_mf = {
    "parametros": parametros_mf.copy(),
    "t": tiempos_mf,
    "estado": historia_mf,
    "alpha": alpha_mf,
    "fotones": fotones_mf,
    "tasa_relativa": tasas_mf,
    "t_estacionario": None,
    "estado_estacionario": None,
}

longitud_espin_mf = float(np.linalg.norm(q0_mf[2:]))
if longitud_espin_mf > 0:
    lambda_critica_mf = np.sqrt(
        omega_0_mf * (omega_mf**2+(kappa_mf/2)**2)
        / (8*longitud_espin_mf*omega_mf)
    )
else:
    lambda_critica_mf = np.inf
error_longitud_mf = np.max(np.abs(
    np.linalg.norm(historia_mf[:, 2:], axis=1)-longitud_espin_mf
))
print(f"Máximo cambio numérico de longitud del espín: {error_longitud_mf:.3e}")
print(f"Umbral del punto normal polarizado: lambda_c = {lambda_critica_mf:.6f}")

if candidatos_mf.size:
    indice_mf = int(candidatos_mf[0])
    t_estacionario_mf = float(tiempos_mf[indice_mf])
    estado_estacionario_mf = historia_mf[indice_mf].copy()

    resultado_mf["t_estacionario"] = t_estacionario_mf
    resultado_mf["estado_estacionario"] = estado_estacionario_mf

    us, vs, xs, ys, zs = estado_estacionario_mf
    print(f"Tiempo estacionario aproximado: {t_estacionario_mf:g}")
    print("Punto fijo aproximado [u, v, x, y, z]:")
    print(estado_estacionario_mf)
    print(f"<a> ≈ {np.sqrt(N_mf) * (us + 1j * vs)}")
    print(f"<Jx>, <Jy>, <Jz> ≈ {N_mf * np.array([xs, ys, zs])}")
    print(f"N |alpha|² ≈ {N_mf * (us**2 + vs**2):.8f}")
else:
    print(
        f"No se confirmó estacionariedad hasta t={t_max_mf:g}. "
        f"Tasa final: {tasas_mf[-1]:.3e}. "
        "Aumenta t_max_mf si quieres continuar la búsqueda."
    )

fig, ax = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

ax[0].plot(tiempos_mf, fotones_mf)
ax[0].set(
    xlabel="Tiempo",
    ylabel=r"$N|\alpha|^2$",
    title="Campo de cavidad en campo medio",
)

for valores, etiqueta in [(x_mf, "x"), (y_mf, "y"), (z_mf, "z")]:
    ax[1].plot(tiempos_mf, valores, label=etiqueta)
ax[1].set(xlabel="Tiempo", ylabel="Espín normalizado", title="Espín colectivo")
ax[1].legend()

ax[2].semilogy(tiempos_mf, np.maximum(tasas_mf, 1e-300))
ax[2].axhline(tol_tasa_mf, color="red", linestyle="--", label="Tolerancia")
ax[2].set(
    xlabel="Tiempo",
    ylabel="Tasa relativa de cambio",
    title="Estacionariedad",
)
ax[2].legend()

for eje in ax:
    if t_estacionario_mf is not None:
        eje.axvline(t_estacionario_mf, color="green", linestyle=":")
    eje.grid(alpha=0.3)

plt.show()


## Fluctuaciones alrededor del punto obtenido por integración

Analizamos el punto numérico mediante el mismo Jacobiano que usan las
funciones de barrido. Comprobamos primero su tasa residual.

Para cada polo $s=-\gamma+i\Omega$, $|\Omega|$ es frecuencia angular
y $\gamma$ es amortiguamiento de amplitud. Si las frecuencias coinciden,
mostramos los modos sin identificarlos arbitrariamente como LP y UP.

La evolución lineal de una pequeña perturbación es

$$\delta\mathbf q(t)=B e^{M_{\mathrm{red}}t}B^T\delta\mathbf q(0).$$

Solo se utiliza el punto cuando la integración confirmó estacionariedad
y el espín medio es no nulo. Si no se encontró, esta sección informa de
ello y permite continuar con los puntos algebraicos del barrido.


In [ ]:
resultado_modos_mf = None
if estado_estacionario_mf is None or np.linalg.norm(estado_estacionario_mf[2:]) < 1e-12:
    print("No hay punto estacionario integrado con espín no nulo para analizar.")
else:
    resultado_modos_mf = analizar_punto_campo_medio(
        estado_estacionario_mf, omega_mf, omega_0_mf, lam_mf, kappa_mf
    )
    tasa_punto_mf = resultado_modos_mf["residuo"] / max(np.linalg.norm(estado_estacionario_mf), 1e-15)
    print(f"Tasa relativa en el punto estacionario: {tasa_punto_mf:.3e}")
    print("Estabilidad:", resultado_modos_mf["estabilidad"])
    print("Autovalores dinámicos:")
    for polo in sorted(resultado_modos_mf["autovalores"], key=lambda s: s.imag):
        print(f"s = {polo.real:+.8f} {polo.imag:+.8f}j")
    if resultado_modos_mf["frecuencia_degenerada"]:
        print("Frecuencias coincidentes: no se asignan los decaimientos a LP/UP.")
        for i, modo in enumerate(resultado_modos_mf["modos"], 1):
            print(f"Modo {i}: Omega={modo['omega']:.8f}, tasas={modo['gammas']}")
    else:
        for nombre, modo in resultado_modos_mf["ramas"].items():
            print(f"{nombre}: Omega={modo['omega']:.8f}, tasas={modo['gammas']}")

    base_fluct_mf = resultado_modos_mf["base"]
    matriz_fluct_mf = resultado_modos_mf["matriz_reducida"]
    delta_q0_mf = np.array([1e-5, 0, 0, 0, 0], dtype=float)
    t_fluctuaciones_mf = np.linspace(0, 60, 1201)
    eta_mf = expm_multiply(
        matriz_fluct_mf, base_fluct_mf.T @ delta_q0_mf,
        start=0, stop=60, num=1201, endpoint=True,
        traceA=np.trace(matriz_fluct_mf),
    )
    delta_q_mf = eta_mf @ base_fluct_mf.T
    fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for col, etiqueta in [(0, r"$\delta u$"), (1, r"$\delta v$")]:
        ax[0].plot(t_fluctuaciones_mf, delta_q_mf[:, col], label=etiqueta)
    for col, etiqueta in [(2, r"$\delta x$"), (3, r"$\delta y$"), (4, r"$\delta z$")]:
        ax[1].plot(t_fluctuaciones_mf, delta_q_mf[:, col], label=etiqueta)
    for eje in ax:
        eje.set(xlabel="Tiempo desde la perturbación", ylabel="Fluctuación")
        eje.legend()
        eje.grid(alpha=0.3)
    plt.show()


## Parámetros y ejemplo para un acoplamiento

Elegimos el número de átomos, las frecuencias y la pérdida de fotones.
Estos parámetros se mantienen fijos durante el barrido en lambda.

Primero evaluamos un único acoplamiento para comprobar el punto
estacionario seleccionado, su estabilidad y sus modos.

Con fase="auto", la función elige el punto normal debajo del umbral
y una rama superradiante encima. Las dos ramas superradiantes tienen
el mismo espectro de pequeñas fluctuaciones.

Los parámetros son independientes de la integración anterior. Para usar
exactamente sus valores, reemplaza este diccionario por una copia de
`parametros_mf` sin la clave `lam`, y añade
`longitud_espin=longitud_espin_mf`.


In [ ]:
# Modifica estos valores para usar los parámetros que quieras.
parametros_barrido = {
    "N": 20,
    "omega": 1.0,
    "omega_0": 1.0,
    "kappa": 0.10,
    "longitud_espin": 0.5,
}

# Ejemplo para un solo lambda.
punto_mf = puntos_y_modos_campo_medio(
    lam=0.5,
    **parametros_barrido
)

print(f"lambda_c = {punto_mf['lambda_c']:.8f}")
print("Rama seleccionada:", punto_mf["rama_elegida"])
print("[u, v, x, y, z] =", punto_mf["seleccionado"]["estado"])
print("Estabilidad:", punto_mf["seleccionado"]["estabilidad"])
print("Residuo estacionario:", punto_mf["seleccionado"]["residuo"])

if punto_mf["seleccionado"]["frecuencia_degenerada"]:
    print("Frecuencias coincidentes; los decaimientos no tienen etiqueta LP/UP única.")
    for i, modo in enumerate(punto_mf["seleccionado"]["modos"], 1):
        print(i, "Omega =", modo["omega"], "tasas gamma =", modo["gammas"])
else:
    for nombre, modo in punto_mf["seleccionado"]["ramas"].items():
        print(nombre, "Omega =", modo["omega"], "tasas gamma =", modo["gammas"])


In [ ]:
lambda_max_mf = 1.0
lambda_c_mf = puntos_y_modos_campo_medio(lam=0, **parametros_barrido)["lambda_c"]
lambdas_mf = malla_con_critico(0, lambda_max_mf, lambda_c_mf)
curvas_mf = barrer_lambda_campo_medio(lambdas_mf, **parametros_barrido)
print(f"Barrido calculado: {len(lambdas_mf)} valores de lambda.")
avisos_barrido_mf(curvas_mf)


## Frecuencias y decaimientos frente a lambda

Las líneas continuas muestran los decaimientos de modos oscilatorios
con frecuencias distintas. Las discontinuas muestran las dos tasas de
una rama no oscilatoria. Los puntos grises muestran las tasas de dos
modos con frecuencia coincidente: no tienen asignación LP/UP única.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for columna, nombre in enumerate(("LP", "UP")):
    axes[0, columna].plot(curvas_mf["lambda"], curvas_mf[f"omega_{nombre}"])
    axes[0, columna].set(title=f"Frecuencia {nombre}", ylabel=r"$\Omega$")
    axes[1, columna].plot(curvas_mf["lambda"], curvas_mf[f"gamma_{nombre}"], label="Oscilatorio, no degenerado")
    mascara_real = curvas_mf[f"no_oscilatoria_{nombre}"]
    if np.any(mascara_real):
        for j in range(2):
            axes[1, columna].plot(curvas_mf["lambda"], np.where(
                mascara_real, curvas_mf[f"gammas_{nombre}"][:, j], np.nan
            ), "--", label=f"Polo real {j+1}")
    if np.any(curvas_mf["degenerada"]):
        for j in range(2):
            axes[1, columna].plot(curvas_mf["lambda"], curvas_mf["gamma_degeneradas"][:, j],
                                  ".", color="gray", label="Sin etiqueta LP/UP" if j == 0 else None)
    axes[1, columna].set(title=f"Decaimiento {nombre}", ylabel=r"$\gamma$")
    axes[1, columna].legend()
for eje in axes.flat:
    eje.set_xlabel(r"$\lambda$")
    if curvas_mf["lambda_c"] <= curvas_mf["lambda"][-1]:
        eje.axvline(curvas_mf["lambda_c"], color="black", linestyle=":")
    eje.grid(alpha=0.3)
plt.show()


## Energías características frente a lambda

Representamos $\epsilon_\pm=\hbar\Omega_{\mathrm{LP/UP}}$, con hbar=1.
Son frecuencias de polos dinámicos expresadas como energías, no los
autovalores exactos del Hamiltoniano finito. En modos muy amortiguados
no deben interpretarse automáticamente como energías de excitaciones
bien definidas ni como máximos del espectro medido.


In [ ]:
hbar = 1.0
epsilon_menos = hbar * curvas_mf["omega_LP"]
epsilon_mas = hbar * curvas_mf["omega_UP"]
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.plot(curvas_mf["lambda"], epsilon_menos, label=r"$\hbar\Omega_{\mathrm{LP}}$")
ax.plot(curvas_mf["lambda"], epsilon_mas, label=r"$\hbar\Omega_{\mathrm{UP}}$")
if 0 <= curvas_mf["lambda_c"] <= curvas_mf["lambda"][-1]:
    ax.axvline(curvas_mf["lambda_c"], color="black", linestyle=":", label=r"$\lambda_c$")
ax.set(xlabel=r"Acoplamiento $\lambda$", ylabel="Energía característica del modo",
       title="Frecuencias de polos dinámicos multiplicadas por hbar")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## Frecuencias características en función de kappa

Mantenemos lambda y las frecuencias fijos y calculamos puntos estacionarios
y modos para cada kappa. Graficamos $\epsilon_\pm=\hbar\Omega_{\mathrm{LP/UP}}$.
Son frecuencias de polos dinámicos multiplicadas por hbar, no niveles
exactos del Hamiltoniano cuántico ni necesariamente picos espectrales.

Para longitud de espín r, si el radicando es positivo:

$$\kappa_c=2\sqrt{8r\lambda^2\omega/\omega_0-\omega^2}.$$

La superradiancia de campo medio corresponde a kappa < kappa_c.
Omega=0 significa ausencia de oscilación: no equivale por sí solo al
umbral, que exige un autovalor completo cero.


In [ ]:
parametros_kappa = {"N": 20, "omega": 1.0, "omega_0": 1.0, "longitud_espin": 0.5}
lambda_fija = 0.60
kappa_max = 2.0
hbar = 1.0
referencia_kappa = barrer_kappa_campo_medio([0.0], lam=lambda_fija, **parametros_kappa)
kappa_c = referencia_kappa["kappa_c"]
kappas = malla_con_critico(0, kappa_max, kappa_c)
curvas_kappa_mf = barrer_kappa_campo_medio(kappas, lam=lambda_fija, **parametros_kappa)
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.plot(kappas, hbar*curvas_kappa_mf["omega_LP"], label=r"$\hbar\Omega_{\mathrm{LP}}$")
ax.plot(kappas, hbar*curvas_kappa_mf["omega_UP"], label=r"$\hbar\Omega_{\mathrm{UP}}$")
if kappa_c is not None and kappa_c <= kappa_max:
    ax.axvline(kappa_c, color="black", linestyle=":", label=r"$\kappa_c$")
ax.set(xlabel=r"Pérdida de fotones $\kappa$", ylabel="Energía característica del modo",
       title=rf"Polos polaritónicos: $\lambda={lambda_fija:g}$")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print("Umbral kappa_c:", kappa_c)
avisos_barrido_mf(curvas_kappa_mf)


## Rama LP frente a lambda para distintas pérdidas

Graficamos la energía de excitación

$$
\epsilon_{\mathrm{LP}}=\hbar\Omega_{\mathrm{LP}}
$$

en función de lambda para diez valores de kappa, identificados mediante
distintos colores. Usamos hbar = 1.

Para cada combinación de parámetros recalculamos el punto estacionario
de campo medio y sus modos, seleccionando automáticamente la fase.

Una frecuencia cero corresponde a un modo no oscilatorio. Los huecos
indican puntos sin asignación automática de la rama LP.

Las etiquetas son locales por frecuencia. En regiones degeneradas las
curvas de frecuencia coinciden, pero no se asigna arbitrariamente su
decaimiento a LP/UP. Estas gráficas no son niveles cuánticos exactos.


In [ ]:
parametros_lp_lambda = {"N": 20, "omega": 1.0, "omega_0": 1.0, "longitud_espin": 0.5}
hbar = 1.0
kappas_comparacion = np.linspace(0.0, 2.0, 10)
lambda_max = 1.0
colores = plt.cm.viridis(np.linspace(0.05, 0.95, 10))
curvas_lp_por_kappa = []
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for kappa_actual, color in zip(kappas_comparacion, colores):
    referencia_lp = puntos_y_modos_campo_medio(
        lam=0, kappa=float(kappa_actual), **parametros_lp_lambda
    )
    malla_lambda = malla_con_critico(0, lambda_max, referencia_lp["lambda_c"])
    resultado_barrido_lambda = barrer_lambda_campo_medio(
        malla_lambda, kappa=float(kappa_actual), **parametros_lp_lambda
    )
    epsilon_lp_lambda = hbar*resultado_barrido_lambda["omega_LP"]
    curvas_lp_por_kappa.append({"kappa": float(kappa_actual), "lambda": malla_lambda,
                              "epsilon_LP": epsilon_lp_lambda, "resultado": resultado_barrido_lambda})
    ax.plot(malla_lambda, epsilon_lp_lambda, color=color, label=rf"$\kappa={kappa_actual:.3f}$")
ax.set(xlabel=r"Acoplamiento $\lambda$", ylabel=r"$\hbar\Omega_{\mathrm{LP}}$",
       title="Frecuencia característica LP para diez pérdidas")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
ax.grid(alpha=0.3)
plt.show()


## Rama LP frente a kappa para distintos acoplamientos

Graficamos la energía de excitación de LP en función de kappa para
diez valores de lambda, identificados mediante distintos colores.

Para cada combinación recalculamos el punto estacionario de campo medio
y sus modos. Usamos hbar = 1.

Una frecuencia cero corresponde a un modo no oscilatorio. Los huecos
indican puntos sin asignación automática de la rama LP.

Las etiquetas son locales por frecuencia. En regiones degeneradas las
curvas de frecuencia coinciden, pero no se asigna arbitrariamente su
decaimiento a LP/UP. Estas gráficas no son niveles cuánticos exactos.


In [ ]:
parametros_lp_kappa = {"N": 20, "omega": 1.0, "omega_0": 1.0, "longitud_espin": 0.5}
hbar = 1.0
lambdas_comparacion = np.linspace(0.1, 1.0, 10)
kappa_max = 2.0
colores = plt.cm.viridis(np.linspace(0.05, 0.95, 10))
curvas_lp_por_lambda = []
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for lambda_actual, color in zip(lambdas_comparacion, colores):
    referencia_lp_kappa = barrer_kappa_campo_medio(
        [0.0], lam=float(lambda_actual), **parametros_lp_kappa
    )
    malla_kappa = malla_con_critico(0, kappa_max, referencia_lp_kappa["kappa_c"])
    resultado_barrido_kappa = barrer_kappa_campo_medio(
        malla_kappa, lam=float(lambda_actual), **parametros_lp_kappa
    )
    epsilon_lp_kappa = hbar*resultado_barrido_kappa["omega_LP"]
    curvas_lp_por_lambda.append({"lambda": float(lambda_actual), "kappa": malla_kappa,
                               "epsilon_LP": epsilon_lp_kappa, "resultado": resultado_barrido_kappa})
    ax.plot(malla_kappa, epsilon_lp_kappa, color=color, label=rf"$\lambda={lambda_actual:.3f}$")
ax.set(xlabel=r"Pérdida de fotones $\kappa$", ylabel=r"$\hbar\Omega_{\mathrm{LP}}$",
       title="Frecuencia característica LP para diez acoplamientos")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
ax.grid(alpha=0.3)
plt.show()
